# BiSI synthesis parameter extraction

Pulls synthesis records from the configured BiSI NOMAD upload IDs (token in `.env`), restricted to `INLThinFilmStack` samples. Falls back to a BiSI-specific local archive export, then to the BiSI synthesis CSV. Writes the extracted database to `data/processed/bisi_synthesis_database_nomad.csv`, keeping the existing process-parameter schema where those fields are present. This notebook is the only pipeline stage migrated and run in this pass; XRD references and optimization remain pending the BiSI CIF/RTF files.

In [6]:
import sys
from pathlib import Path
import importlib

# Make the package importable when the kernel runs from the notebooks/ folder.
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from xrd_bo_optimization_tf.config import (
    NOMAD_STAGING_DIR,
    NOMAD_SYNTHESIS_CSV,
    NOMAD_VOLUME_DIR,
    SYNTHESIS_CSV,
)
from xrd_bo_optimization_tf.nomad_client import iter_archive_files
import xrd_bo_optimization_tf.synthesis as synthesis_module

# Reload so a long-lived notebook kernel uses the current extractor implementation.
importlib.reload(synthesis_module)

build_synthesis_dataframe = synthesis_module.build_synthesis_dataframe
load_synthesis_archives = synthesis_module.load_synthesis_archives
load_synthesis_from_nomad = synthesis_module.load_synthesis_from_nomad

# Prefer the live NOMAD API. Fall back to a BiSI-specific local archive export, then a CSV.
try:
    synthesis_db = load_synthesis_from_nomad(NOMAD_STAGING_DIR)
    data_source = f"NOMAD API ({len(synthesis_db)} samples)"
except Exception as nomad_error:  # noqa: BLE001 - offline fallback
    print(f"NOMAD API unavailable ({nomad_error}); trying local archives.")
    archive_paths = list(iter_archive_files(NOMAD_VOLUME_DIR))
    synthesis_db = load_synthesis_archives(NOMAD_VOLUME_DIR) if archive_paths else {}
    data_source = f"{len(archive_paths)} local NOMAD archive files"

if synthesis_db:
    df = build_synthesis_dataframe(synthesis_db)
    NOMAD_SYNTHESIS_CSV.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(NOMAD_SYNTHESIS_CSV, index=False)
    print(f"Wrote complete synthesis table to {NOMAD_SYNTHESIS_CSV}")
elif SYNTHESIS_CSV.is_file():
    df = pd.read_csv(SYNTHESIS_CSV)
    data_source = str(SYNTHESIS_CSV)
else:
    raise FileNotFoundError(
        f"No NOMAD source reachable and no BiSI synthesis CSV at {SYNTHESIS_CSV}"
    )

print(f"Loaded {len(df)} synthesis records from {data_source}")
df.head()

Wrote complete synthesis table to /home/nomad-lanasc/bisi-xrd-optimization/data/processed/bisi_synthesis_database_nomad.csv
Loaded 41 synthesis records from NOMAD API (41 samples)


,sample_id,series,completeness,spin_speed_rpm,spin_duration_s,spin_acceleration,soft_bake_temp_c,soft_bake_duration_s,soft_bake_duration_min,hard_bake_temp_c,...,i_to_bi_molar_ratio,seed_solution_name,seed_solution_chemistry_name,seed_bismuth_concentration_mol_l,seed_s_to_bi_molar_ratio,seed_i_to_bi_molar_ratio,n_spin_cycles,spin_to_se_days,num_archives,has_annealing
0,260408A1,260408A,complete,3000.0,30.0,1000.0,150.0,180.0,3.0,170.0,...,1.046516,260408 DillutedBiSI 0.05 milliliter,260408 DillutedBiSI,0.106773,0.982289,1.046516,2,None,1,True
1,260408A2,260408A,complete,3000.0,30.0,1000.0,150.0,180.0,3.0,190.0,...,1.046516,260408 DillutedBiSI 0.05 milliliter,260408 DillutedBiSI,0.106773,0.982289,1.046516,2,None,1,True
2,260408A3,260408A,complete,3000.0,30.0,1000.0,150.0,180.0,3.0,210.0,...,1.046516,260408 DillutedBiSI 0.05 milliliter,260408 DillutedBiSI,0.106773,0.982289,1.046516,2,None,1,True
3,260408A4,260408A,complete,3000.0,30.0,1000.0,150.0,180.0,3.0,150.0,...,1.046516,260408 DillutedBiSI 0.05 milliliter,260408 DillutedBiSI,0.106773,0.982289,1.046516,3,None,1,True
4,260408A5,260408A,complete,3000.0,30.0,1000.0,150.0,180.0,3.0,230.0,...,1.046516,260408 DillutedBiSI 0.05 milliliter,260408 DillutedBiSI,0.106773,0.982289,1.046516,2,None,1,True


In [7]:
# Audit availability of BiSI precursor chemistry in the main deposition and seed solutions.
chemistry_columns = [
    "bismuth_concentration_mol_l",
    "s_to_bi_molar_ratio",
    "i_to_bi_molar_ratio",
]
chemistry_available = [column for column in chemistry_columns if column in df]
completeness = (
    df[chemistry_available]
    .notna()
    .sum()
    .rename("available_samples")
    .to_frame()
)
completeness["total_samples"] = len(df)

seed_columns = [f"seed_{column}" for column in chemistry_columns]
seed_available = [column for column in seed_columns if column in df]
if seed_available:
    seed_completeness = (
        df[seed_available]
        .notna()
        .sum()
        .rename("available_samples")
        .to_frame()
    )
    seed_completeness["total_samples"] = len(df)
    seed_completeness.index = [column.removeprefix("seed_") + " (seed)" for column in seed_completeness.index]
    completeness = pd.concat([completeness, seed_completeness])

completeness

,available_samples,total_samples
bismuth_concentration_mol_l,39,41
s_to_bi_molar_ratio,39,41
i_to_bi_molar_ratio,39,41
bismuth_concentration_mol_l (seed),41,41
s_to_bi_molar_ratio (seed),41,41
i_to_bi_molar_ratio (seed),41,41


In [4]:
from xrd_bo_optimization_tf.nomad_api import load_upload_ids

try:
    upload_ids = load_upload_ids()
    print(f'{len(upload_ids)} BiSI uploads configured:')
    for upload_id in upload_ids:
        print(f'  {upload_id}')
except Exception as error:  # noqa: BLE001
    print(f'Upload-ID list unavailable: {error}')

1 BiSI uploads configured:
  HnDaDBQhT1a7IPwy5NoSOg
